In [31]:
using DataStructures
using Distributions

mutable struct TabuState{TMove,P,TF}
    tabu_buffer::CircularBuffer{TMove}
    best_seen::P
    best_seen_obj::TF
    current::P
    considered::P
    iter::Int
end

function TabuState(p, x0; buffer_length::Int=10)
    moves = possible_moves(p, x0)
    obj = objective(p, x0)
    return TabuState{eltype(moves),typeof(x0),typeof(obj)}(
        CircularBuffer{eltype(moves)}(buffer_length), x0, obj, copy(x0), copy(x0), 1
    )
end


function solve_tabu(p, s::TabuState; iteration_limit::Int=100)
    while s.iter < iteration_limit
        moves = possible_moves(p, s.current)
        best_move = 0
        best_move_obj = Inf
        for (i_move, move) in enumerate(moves)
            if in(move, s.tabu_buffer)
                # move forbidden, do not consider
                continue
            end
            # evaluate move
            copyto!(s.considered, s.current)
            apply!(s.considered, move)
            considered_value = objective(p, s.considered)
            if considered_value < best_move_obj
                best_move = i_move
                best_move_obj = considered_value
            end
        end
        # no allowed move found
        if best_move == 0
            break
        end
        chosen_move = moves[best_move]
        item_id = chosen_move[1]
        prev_bin = s.current[item_id]
        apply!(s.current, chosen_move)
        push!(s.tabu_buffer, invert_move(p, moves[best_move], prev_bin))
        if best_move_obj < s.best_seen_obj
            # best so far, let's remember it
            copyto!(s.best_seen, s.current)
            s.best_seen_obj = best_move_obj
        end
        s.iter += 1
    end
    return s.best_seen
end


struct KnapsackProblem
    capacities::Vector{Int}
    weights::Vector{Int}
    profits::Vector{Int}
end

function objective(p::KnapsackProblem, x::Vector{Int})
    profit = 0
    for i in eachindex(x)
        if x[i] > 0
            profit += p.profits[i]
        end
    end
    return -profit
end


function apply!(x, move::Tuple{Int,Int})
    item_idx, new_bin = move
    x[item_idx] = new_bin
    return x
end

function invert_move(::KnapsackProblem, move::Tuple{Int,Int}, old_bin::Int)
    return (move[1], old_bin)
end


function possible_moves(p::KnapsackProblem, x::Vector{Int})
    move_list = Tuple{Int,Int}[]
    current_weights = zeros(Int, length(p.capacities)) #sum(p.weights .* x)
    # add item
    for (item_id, bin_id) in enumerate(x)
        if bin_id > 0 # 0 oznacza brak przypisania
            current_weights[bin_id] += p.weights[item_id]
        end
    end    

    for i in eachindex(x)
        if x[i] > 0
            # możliwość usunięcia i z plecaka x[i]
            push!(move_list, (i, 0))
        end
        for k in eachindex(p.capacities)
            # przeniesienie do plecaka
            if x[i] == 0 && (current_weights[k] + p.weights[i] <= p.capacities[k])
                push!(move_list, (i,k))
            end
            # dodatkowa możliwość - przeniesienie do innego plecaka, do punktu 4.0
            # if x[i] != k && (current_weights[k] + p.weights[i] <= p.capacities[k])
            #     push!(move_list, (i, k))
            # end
        end
    end

    return move_list
end



possible_moves (generic function with 1 method)

In [33]:
mutable struct SAState{P, TF}
    current::P
    best_seen::P
    best_seen_obj::TF
    temp::Float64
    iter::Int
end

function solve_simulated_annealing(p, s::SAState; 
                                   max_iter=10000, 
                                   cooling_rate=0.995)
    
    while s.iter < max_iter
        moves = possible_moves(p, s.current)
        if isempty(moves) break end
        move = rand(moves) 
        old_obj = objective(p, s.current)
        
        test_state = copy(s.current)
        apply!(test_state, move)
        new_obj = objective(p, test_state)
        
        ΔE = new_obj - old_obj
        
        if ΔE < 0 || rand() < exp(-ΔE / s.temp)
            s.current = test_state
            
            if new_obj < s.best_seen_obj
                s.best_seen = copy(s.current)
                s.best_seen_obj = new_obj
            end
        end
        
        s.temp *= cooling_rate
        s.iter += 1
    end
    return s.best_seen
end

solve_simulated_annealing (generic function with 1 method)

In [27]:

function generate_problem(n_knapsacks = 3, n_items = 100)
    capacities = rand(DiscreteUniform(2500, 3000), n_knapsacks)
    profits = rand(DiscreteUniform(10, 1000), n_items)
    weights = rand(DiscreteUniform(10, 100), n_items)
    kp = KnapsackProblem(capacities, profits, weights)
end

kp1 = generate_problem()


KnapsackProblem([2826, 2944, 2941], [496, 185, 176, 705, 922, 429, 445, 18, 670, 924  …  739, 513, 964, 579, 742, 942, 785, 606, 961, 697], [36, 44, 52, 57, 31, 96, 55, 81, 82, 83  …  66, 49, 23, 26, 28, 41, 41, 47, 25, 43])

In [ ]:
function test(kp)
    x0 = fill(0, length(kp.weights))
    st = TabuState(kp, x0; buffer_length=10)
    sol = solve_tabu(kp, st; iteration_limit=1000000)

    println("Pełne rozwiązanie: ", sol)
    
    println("Best objective: ", st.best_seen_obj)
    println("Last iteration: ", st.iter)
end
@time begin
test(kp1)
end


Pełne rozwiązanie: [0, 0, 3, 0, 0, 1, 0, 2, 0, 3, 3, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 3, 2, 0, 2, 0, 0, 0, 1, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 3, 1, 0, 0, 1, 0, 3, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0, 3, 2, 1, 0, 0, 0, 0, 0, 3, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Best objective: -2020
Last iteration: 1000000
  1.850854 seconds (6.13 M allocations: 906.807 MiB, 5.45% gc time, 3.97% compilation time)


In [34]:
function test2(kp)
    x0 = fill(0, length(kp.weights))
    st = SAState(kp, x0; buffer_length=10)
    sol = solve_simulated_annealing(kp, st; iteration_limit=1000000)

    println("Full solution: ", sol)
    
    println("Best objective: ", st.best_seen_obj)
    println("Last iteration: ", st.iter)
end
@time begin
test(kp1)
end

Pełne rozwiązanie: [0, 0, 3, 0, 0, 1, 0, 2, 0, 3, 3, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 3, 2, 0, 2, 0, 0, 0, 1, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 3, 1, 0, 0, 1, 0, 3, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 0, 3, 2, 1, 0, 0, 0, 0, 0, 3, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
Best objective: -2020
Last iteration: 1000000
  1.799873 seconds (6.00 M allocations: 900.467 MiB, 5.56% gc time)


In [25]:
function experiment(kp)
    dlugosci = [1, 2, 5]
    
    for L in dlugosci
        @time begin
        println("\n--- Eksperyment dla długości tabu L = $L ---")
        
        x0 = fill(0, length(kp.weights))
        # Tutaj przekazujemy L do parametru buffer_length
        st = TabuState(kp, x0; buffer_length = L)
        
        sol = solve_tabu(kp, st; iteration_limit = 100000)
        
        println("Pełne rozwiązanie: ", sol)
    
        println("Best objective: ", st.best_seen_obj)
        println("Last iteration: ", st.iter)
        end
    end
end
experiment(kp1)


--- Eksperyment dla długości tabu L = 1 ---
Pełne rozwiązanie: [3, 1, 0, 3, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 3, 0, 0, 1, 0, 0, 0, 0, 1, 0, 2, 0, 0, 0, 1, 3, 0, 0, 0, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 1, 0, 1, 0, 0, 0, 0, 0]
Best objective: -1704
Last iteration: 100000
  0.154837 seconds (601.08 k allocations: 90.218 MiB, 10.12% gc time)

--- Eksperyment dla długości tabu L = 2 ---
Pełne rozwiązanie: [3, 1, 0, 3, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 3, 0, 0, 1, 0, 0, 0, 0, 1, 0, 2, 0, 0, 0, 1, 3, 0, 0, 0, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2, 1, 0, 1, 0, 0, 0, 0, 0]
Best objective: -1704
Last iteration: 100000
  0.147987 seconds (601.03 k allocations: 90.217 MiB, 5.57% gc time)

--- Eksperyment dla długości tabu L = 5 